<a href="https://colab.research.google.com/github/CodeByQasim/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook & Recommendation Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Translating cluster archetypes into prioritized editorial workflows.

## 1. Operational Playbook Rules

Each archetype maps to an operational protocol with explicit reason codes:

1. **Protect & Monitor (8,298 items, 27.7%):** High impression authority drivers (median 7,896 imp, rank 8.6). *Action:* Lock URL slug, audit internal links, monitor monthly position drops.
2. **Improve Snippets & Metadata (11,783 items, 39.3%):** Striking-distance inventory (median 851 imp, rank 15.5 on Page 2, CTR 0.07%). *Action:* Rewrite title tags, add FAQ schema, improve meta description CTR hooks.
3. **Boost Internal Linking (6,962 items, 23.2%):** Younger long-tail articles (median 32 imp, rank 10.1, 20% scroll). *Action:* Link from top authority articles to transfer PageRank.
4. **Non-Keyword Feedly Strategy (1,749 items, 5.8%):** Non-search articles (97% missing keyword volume). *Action:* Assign target search keywords and optimize headings.
5. **Merge or Prune (1,208 items, 4.0%):** Zombie content (median 1 imp, unranked). *Action:* 301 redirect to pillar content or 410 prune to save crawl budget.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Auto-detect data path: works in local repo or directly in Google Colab
DATA_URL = 'https://raw.githubusercontent.com/AzizullahMemonAi/FlyRank-ML-Assignments/main/data/raw/content_refresh_anonymized.csv'
LOCAL_PATHS = [
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv')
]
data_path = next((p for p in LOCAL_PATHS if p.exists()), None)
df = pd.read_csv(data_path if data_path is not None else DATA_URL)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Generate ranked queue
archetype_names = {
    0: 'Authority Drivers',
    1: 'Striking-Distance Opportunity',
    2: 'Emerging Long-Tail',
    3: 'Zombie / Unranked',
    4: 'Feedly Niche Editorial'
}
action_names = {
    0: 'Protect & Monitor',
    1: 'Improve Metadata & CTR',
    2: 'Boost Internal Links',
    3: 'Merge or Prune',
    4: 'Keyword Refresh'
}
print('Playbook Action Distribution across 30,000 pages:')
print(pd.Series(action_names).to_string())


Playbook Action Distribution across 30,000 pages:
0         Protect & Monitor
1    Improve Metadata & CTR
2      Boost Internal Links
3            Merge or Prune
4           Keyword Refresh


## 2. Priority Ranking Score

Within each cluster, pages are ranked by **Opportunity Impact Score** $= \log(1 + \text{impressions}) \times (1 - \text{CTR}) \times (1 / \text{rank})$, bubbling up high-exposure pages with immediate fix potential.